# Adversarial robustness — clinical cohort

Run the same robustness battery on the small clinical cohort (insurance prediction). n is small so the numbers are noisy — that is the point of this test.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from examples.adversarial import clinical_binary_dataset, train_test, train, robustness_sweep

X, y, feats, target = clinical_binary_dataset(300, seed=7)
Xtr, Xte, ytr, yte = train_test(X, y)
print('features:', feats, '| target:', target)
print('train/test:', Xtr.shape[0], '/', Xte.shape[0])
print('class balance: %.2f' % y.mean())

In [ ]:
model = train('mlp', Xtr, ytr)
print('clean accuracy: %.3f' % (model.predict(Xte) == yte).mean())
epsilons = [0.0, 0.1, 0.25, 0.5, 1.0]
df = pd.DataFrame(robustness_sweep(model, Xte, yte, epsilons))
print(df.to_string(index=False))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df['eps'], df['clean_accuracy'], 'o-', label='clean accuracy', color='#4f8cff')
ax.plot(df['eps'], df['robust_accuracy'], 'o-', label='robust accuracy', color='#e05b5b')
ax.plot(df['eps'], df['asr'], 'o--', label='ASR (on correct)', color='#d9a441')
ax.set_xlabel('perturbation budget eps'); ax.set_ylabel('accuracy')
ax.legend(); ax.set_title('Clinical cohort — robustness vs eps (small n)')
ax.grid(alpha=0.3)